# Crop Yield Trend Summary

Reads `agriculture.crop_statistics` (see
`data-generators/usda_crop_statistics_ingest.ipynb`) and computes
state-by-commodity yield trends between the two ingested census years —
an agribusiness/ag-lending question: which state/commodity combinations
are seeing yields rise or fall, to inform territory planning or
crop-insurance risk pricing.

Cleans NASS's raw `VALUE` formatting (comma-separated numbers,
`(D)`/`(Z)`/`(NA)` suppression sentinels) inline, pivots the three
stat categories into columns, then computes the change between census
years via a window function.

Re-runnable: `mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "agriculture"
SOURCE_TABLE = "crop_statistics"
TARGET_TABLE = "crop_yield_trend_summary_v1"

## Load bronze table

In [ ]:
df = spark.table(f"{DATABASE_NAME}.{SOURCE_TABLE}")

## Clean the raw VALUE column

Strips thousands separators and nulls out NASS's suppression sentinels
(`(D)` withheld to avoid disclosure, `(Z)` negligible, `(NA)` not
available) before casting to a number.

In [ ]:
clean_df = (
    df
    .withColumn("value_stripped", F.regexp_replace(F.col("VALUE"), ",", ""))
    .withColumn(
        "value_clean",
        F.when(F.col("value_stripped").rlike(r"^[0-9.]+$"), F.col("value_stripped").cast("double")),
    )
)

## Pivot into area/production/yield columns

In [ ]:
pivoted_df = (
    clean_df
    .groupBy(
        F.col("STATE_ALPHA").alias("state"),
        F.col("COMMODITY_DESC").alias("commodity"),
        F.col("YEAR").alias("census_year"),
    )
    .pivot("STATISTICCAT_DESC", ["AREA HARVESTED", "PRODUCTION", "YIELD"])
    .agg(F.first("value_clean"))
    .withColumnRenamed("AREA HARVESTED", "area_harvested_acres")
    .withColumnRenamed("PRODUCTION", "production")
    .withColumnRenamed("YIELD", "yield_per_acre")
)

## Transform: yield change between census years

In [ ]:
trend_window = Window.partitionBy("state", "commodity").orderBy("census_year")

summary_df = (
    pivoted_df
    .withColumn("prior_census_yield", F.lag("yield_per_acre").over(trend_window))
    .withColumn(
        "yield_change_pct",
        F.round(
            (F.col("yield_per_acre") - F.col("prior_census_yield")) * 100.0 / F.col("prior_census_yield"),
            1,
        ),
    )
    .orderBy("state", "commodity", "census_year")
)

## Persist as a managed Spark table

In [ ]:
summary_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TARGET_TABLE}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_rows FROM {DATABASE_NAME}.{TARGET_TABLE}").show()
spark.sql(
    f"SELECT * FROM {DATABASE_NAME}.{TARGET_TABLE} "
    "WHERE yield_change_pct IS NOT NULL ORDER BY yield_change_pct DESC LIMIT 10"
).show()